In [7]:
%load_ext autoreload
%autoreload 2

import networkx as nx
from pathlib import Path
import pipeline
import pandas as pd
from collections import defaultdict

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### build initial full network

In [ ]:
data = Path('data/undir_trials/24h/depnet_undirected_24h.gz')
df = pd.read_csv(data)
df['DEP'] = pd.to_numeric(df['DEP'], errors='coerce')
df = df.dropna(subset=['DEP'])

filepath = f'data/undir_trials/full_network_24h.txt'
df.to_csv(filepath, sep=' ', index=False, header=False)

print(f"Saved with {len(df)} edges.")

C:\Users\User\AppData\Local\Temp\ipykernel_39508\2852112609.py:2: DtypeWarning: Columns (0: N_UIDS_DESTINATION, 1: N_VISITS_DESTINATION, 2: DEP) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data)


Saved with 15133742 edges.


load in

In [ ]:
gpath = Path('data/undir_trials/full_network_24h.txt')
G = pipeline.load(gpath)

# --- add census tract attribute ---
pipeline.node_to_area(G)
num_empty = sum(1 for n, d in G.nodes(data=True) if d.get('tract') == 'Unknown')
print(f'% empty: {((num_empty/len(G.nodes()))*100):.4f}')

Nodes: 115972
Edges: 15133742
% empty: 9.320353188700722


### map POIs to buckets

In [ ]:
# --- create blank network ---

# collect cats and tracts
cats = set(nx.get_node_attributes(G, 'poi_type').values())
tracts = set(nx.get_node_attributes(G, 'tract').values())
agg_nodes = {(c, t) for c in cats for t in tracts}

print(f'len cats: {len(cats)} -- len tracts: {len(tracts)} -- # combos: {len(agg_nodes)}')

# init network
G_agg = nx.Graph()
G_agg.add_nodes_from(agg_nodes)

# --- map nodes to buckets ---

mapping = defaultdict(list)

for poi, attrs in G.nodes(data=True):
    c = attrs.get('poi_type')
    t = attrs.get('tract')
    
    if (c, t) in G_agg:
        mapping[(c, t)].append(poi)

# prune empty nodes
empty_nodes = [node for node in G_agg.nodes() if node not in mapping]
G_agg.remove_nodes_from(empty_nodes)

len cats: 20 -- len tracts: 1029 -- # combos: 20580


### edges and node attrs

In [14]:
# sum up visits (node attr)
for bucket, poi_list in mapping.items():
    total_visits = sum(G.nodes[poi].get('total_visits', 0) for poi in poi_list)
    G_agg.nodes[bucket]['total_visits'] = total_visits

# --- add edges ---

# create fast lookup
poi_to_bucket = {}
for bucket, poi_list in mapping.items():
    for poi in poi_list:
        poi_to_bucket[poi] = bucket

# add based on lookup (incl attrs)
for u, v, attrs in G.edges(data=True):
    bucket_u = poi_to_bucket.get(u)
    bucket_v = poi_to_bucket.get(v)
    
    if bucket_u and bucket_v and (bucket_u != bucket_v):
        cur_weight = 0
        cur_cov = 0
        new_cov = attrs.get('covisits', 0)
        # check if already exists
        if G_agg.has_edge(bucket_u, bucket_v):
            cur_weight = G_agg[bucket_u][bucket_v].get('weight', 0)
            cur_cov = G_agg[bucket_u][bucket_v].get('covisits', 0)
        # upsert (create or override)
        G_agg.add_edge(bucket_u, bucket_v, weight=cur_weight+1, covisits=cur_cov+new_cov)

# --- recompute dep ---

for i, j, attrs in G_agg.edges(data=True):
    cov = attrs.get('covisits')
    visits_i = G_agg.nodes[i]['total_visits']
    visits_j = G_agg.nodes[j]['total_visits']

    dep = (cov / (visits_i * visits_j))

    attrs['dep'] = dep

# save
nx.write_edgelist(G_agg, 'data/agg_network_24h_undir.txt')

### printout/analysis